# Lunar DEM Reconstruction using Shape-from-Shading

This notebook contains my implementation of a Shape-from-Shading
pipeline developed during the ISRO Bharatiya Antariksh Hackathon 2025.

The reconstruction uses a Lambertian reflectance model to estimate
surface gradients and the Frankot–Chellappa algorithm to integrate
them into a relative height map.

In [ ]:
import numpy as np
from scipy.fft import fft2, ifft2, ifftshift

# --- 1) Frankot–Chellappa Integrator ---

def frankotchellappa(p, q):
    """
    Integrates slopes p, q into height z via Frankot–Chellappa.
    Implements the discrete Poisson integrator in frequency domain.
    """
    m, n = p.shape

    # Frequency coordinates
    u = (np.arange(n) - n//2) / (n - (n % 2))
    v = (np.arange(m) - m//2) / (m - (m % 2))
    wx, wy = np.meshgrid(u, v)

    wx = ifftshift(wx)
    wy = ifftshift(wy)

    # FFT of gradients
    Fp = fft2(p)
    Fq = fft2(q)

    # Avoid division by zero
    denom = wx**2 + wy**2 + np.finfo(float).eps

    # Combine into Z in frequency domain
    Z = (-1j*wx*Fp - 1j*wy*Fq) / denom
    Z[0, 0] = 0  # Zero-mean constraint

    # Inverse FFT to get z
    return np.real(ifft2(Z))


# --- 2) Shape-from-Shading (Discrete Gauss–Seidel) ---

def shape_from_shading_discrete(
    I, sun_vec, rho=0.2, lam=0.2, max_iter=200, tol=1e-4
):
    """
    I         : 2D array, normalized reflectance image [0..1]
    sun_vec   : 3-element array [sx, sy, sz], illumination direction
    rho       : constant Lambertian albedo
    lam       : weight for irradiance term in update
    max_iter  : maximum Gauss-Seidel iterations
    tol       : convergence criterion on maximum update step

    Returns:
        f, g : gradient fields (dz/dx, dz/dy)
        z    : integrated relative height map
    """
    sx, sy, sz = sun_vec / np.linalg.norm(sun_vec)
    m, n = I.shape

    # Initialize gradient estimates to zero
    f = np.zeros((m, n), dtype=float)
    g = np.zeros((m, n), dtype=float)

    # Gauss–Seidel sweep
    for it in range(max_iter):
        max_change = 0.0

        # Update each interior pixel
        for i in range(1, m-1):
            for j in range(1, n-1):

                # Smoothness term: neighbor average
                f_bar = 0.25 * (
                    f[i+1, j] + f[i-1, j] +
                    f[i, j+1] + f[i, j-1]
                )
                g_bar = 0.25 * (
                    g[i+1, j] + g[i-1, j] +
                    g[i, j+1] + g[i, j-1]
                )

                # Compute reflectance and partial derivatives
                A = -f[i, j]*sx - g[i, j]*sy + sz
                D = np.sqrt(1 + f[i, j]**2 + g[i, j]**2)
                R = rho * (A / D)

                # Derivatives of the Lambertian reflectance
                dR_df = rho * (
                    (-sx * D) - A * (f[i, j] / D)
                ) / (D**2)

                dR_dg = rho * (
                    (-sy * D) - A * (g[i, j] / D)
                ) / (D**2)

                # Irradiance residual
                err = I[i, j] - R

                # Update rules
                f_new = f_bar + lam * err * dR_df
                g_new = g_bar + lam * err * dR_dg

                # Track largest update for convergence
                change = max(
                    abs(f_new - f[i, j]),
                    abs(g_new - g[i, j])
                )
                max_change = max(max_change, change)

                # Commit update
                f[i, j], g[i, j] = f_new, g_new

        print(f"Iteration {it+1}: Max Change = {max_change:.6f}")

        if max_change < tol:
            print(f"Converged in {it+1} iterations.")
            break
    else:
        print("Warning: Max iterations reached without full convergence.")

    # Integrate gradients p, q into the height map z
    z = frankotchellappa(f, g)

    return f, g, z
